# Lab D — ROCm GPU Optimization / ROCm GPU 优化
## Slide 43–48 · 输入尺寸与 Batch 的权衡

**What will you learn? / 本节学什么？**
- 输入分辨率如何同时影响延迟、精度与显存 How input resolution affects latency, accuracy and VRAM
- Batch 如何影响吞吐与显存，以及为什么“batch 时间 ÷ 8”不是单请求延迟 How batch affects throughput/VRAM — and why batch-time ÷ 8 is *not* per-request latency

**One key idea / 一个核心原理**
> 计算量 ∝ 分辨率² 、显存 ∝ batch×分辨率²。GPU 优化不是“越大越好”，而是找到匹配业务目标的组合。Compute scales with resolution² and memory > with batch × resolution². GPU tuning is a search for the right trade-off, > not 'bigger is better'.

In [1]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

repo: /workspace/yolo26-openvino-rocm-day2-labs


## Experiment A — Input resolution / 实验A：输入分辨率

**因果链 / causal chain**：输入图片变小 → 模型处理的空间尺寸减小（如 640²→512²，像素少 36%）→ 计算量近似同比下降 → 但小目标的信息可能受损 → mAP 可能下降。
Smaller input → smaller spatial size (640²→512² = 36% fewer pixels) → roughly proportional compute drop → but small-object detail may be lost → mAP may drop.

固定：YOLO26l · W7900 · batch=1 · FP32 · 相同验证图片。唯一变量：`imgsz`。
Fixed: YOLO26l · W7900 · batch=1 · FP32 · same images. Only `imgsz` changes.


In [2]:
import torch
from course.model import load_official_yolo26l
from course.benchmark import bench, save_run, RunMeta, load_bench_images, map5095

model = load_official_yolo26l()
images = load_bench_images(); IMG = images[0]
WARMUP, ITERS = 3, 10

def gpu_bench(imgsz):
    ms = bench(lambda: model.predict(source=str(IMG), imgsz=imgsz,
                           device="cuda", verbose=False),
               warmup=WARMUP, iters=ITERS, sync=True)
    torch.cuda.reset_peak_memory_stats()
    model.predict(source=str(IMG), imgsz=imgsz, device="cuda",
                  verbose=False)
    torch.cuda.synchronize()
    vram = torch.cuda.max_memory_allocated() / 1024**3
    return ms, vram

ms640, vram640 = gpu_bench(640)
sum640 = save_run(RunMeta("labD_resolution", "imgsz640", "cuda(gfx1100)",
                          "pytorch-rocm", imgsz=640, warmup=WARMUP, iters=ITERS,
                          notes="notebook quick run"), ms640)
print(f"imgsz=640 · P50 {sum640['stats']['p50_ms']:.1f} ms · VRAM {vram640:.2f} GB")

[saved] results/raw/labD_resolution/labD_resolution_imgsz640_20261108_112901.csv  (10 samples)
imgsz=640 · P50 12.8 ms · VRAM 0.24 GB


In [3]:
STUDENT_IMGSZ_B = 512           # <-- change me: try 416 / 512 / 768

msB, vramB = gpu_bench(STUDENT_IMGSZ_B)
sumB = save_run(RunMeta("labD_resolution", f"imgsz{STUDENT_IMGSZ_B}",
                        "cuda(gfx1100)", "pytorch-rocm",
                        imgsz=STUDENT_IMGSZ_B, warmup=WARMUP, iters=ITERS,
                        notes="notebook quick run"), msB)
print(f"imgsz={STUDENT_IMGSZ_B} · P50 {sumB['stats']['p50_ms']:.1f} ms · VRAM {vramB:.2f} GB")

[saved] results/raw/labD_resolution/labD_resolution_imgsz512_20261108_112903.csv  (10 samples)
imgsz=512 · P50 13.4 ms · VRAM 0.22 GB


In [4]:
yaml = str(REPO / "data/val_subset.yaml")
map640 = map5095(model, yaml, 640, 8, "cuda")
mapB   = map5095(model, yaml, STUDENT_IMGSZ_B, 8, "cuda")
import pandas as pd
display(pd.DataFrame([
    {"imgsz": 640, "P50 (ms)": round(sum640['stats']['p50_ms'], 1),
     "mAP50-95": round(map640['map50_95'], 4), "VRAM (GB)": round(vram640, 2)},
    {"imgsz": STUDENT_IMGSZ_B, "P50 (ms)": round(sumB['stats']['p50_ms'], 1),
     "mAP50-95": round(mapB['map50_95'], 4), "VRAM (GB)": round(vramB, 2)}]))
from course.charts import chart_resolution
chart_resolution(640, STUDENT_IMGSZ_B, sum640["stats"], sumB["stats"],
                 map640["map50_95"], mapB["map50_95"], vram640, vramB, sumB)

Ultralytics 8.4.207 🚀 Python-3.12.3 torch-2.12.0+rocm7.14.1 CUDA:0 (AMD Radeon Pro W7900D, 49136MiB)


YOLO26l summary (fused): 188 layers, 24,807,420 parameters, 0 gradients, 86.8 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2393.6±853.1 MB/s, size: 63.3 KB)


val: Scanning /workspace/yolo26-openvino-rocm-day2-labs/data/val_subset/labels.cache... 120 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 38.7Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 6% ╸─────────── 1/15 1.9s/it 0.6s<26.4s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 13% ━╸────────── 2/15 3.3it/s 0.7s<3.9s

... (24 outputs left out)

Speed: 0.4ms preprocess, 5.3ms inference, 0.0ms loss, 0.5ms postprocess per image


Results saved to /workspace/yolo26-openvino-rocm-day2-labs/runs/detect/val-23


,imgsz,P50 (ms),mAP50-95,VRAM (GB)
0,640,12.8,0.6398,0.24
1,512,13.4,0.6159,0.22


[chart] /workspace/yolo26-openvino-rocm-day2-labs/assets/slides/04_resolution_vs_latency_accuracy.png (+svg)


## Experiment B — Batch size / 实验B：Batch 大小

**时间轴对比 / timeline intuition**:

```text
batch=1 (串行 8 张):  [img1][img2][img3][img4][img5][img6][img7][img8]   8 × T1
batch=8 (一批 8 张):  [ img1 .. img8 ─── 一批同时算 ─── ] T8 < 8 × T1
                      ↑ 8 个请求必须一起出发（攒批），等待时间属于某个真实用户
```

固定 `imgsz=640`。我们直接测量 **GPU 前向计算** 的整批耗时：
- `batch=1`：单张延迟就是单请求延迟。
- `batch=8`：一整批 8 张一起算。**吞吐 = 8 ÷ 批耗时**；"批耗时 ÷ 8" 只是**单张等效计算成本**，**不是任何单个请求的响应延迟**——请求必须攒够 8 张才能出发。

Fixed `imgsz=640`, we time **GPU forward** for whole batches. batch=1: latency = per-request latency. batch=8: throughput = 8 / batch-time; batch-time/8 is an *equivalent per-image cost*, NOT a per-request latency — requests must wait for the batch to fill.


In [5]:
from course.pipeline import Yolo26lStages
stages = Yolo26lStages(imgsz=640)

def timed_batch(paths):
    import cv2
    im0s = [stages.load(p) for p in paths]
    im = stages.preprocess(im0s)
    torch.cuda.synchronize()
    from course.benchmark import now_ms
    t0 = now_ms(); stages.infer(im); torch.cuda.synchronize()
    return now_ms() - t0

window8 = images[:8]
for _ in range(3):                       # warm-up
    timed_batch(window8); timed_batch(images[:1])

torch.cuda.reset_peak_memory_stats()
b1 = [timed_batch(images[:1]) for _ in range(ITERS)]
b8 = [timed_batch(window8) for _ in range(ITERS)]
vram_batch = torch.cuda.max_memory_allocated() / 1024**3
from course.benchmark import pct
thr1, thr8 = 1000/pct(b1, 50), 8*1000/pct(b8, 50)
sumb1 = save_run(RunMeta("labD_batch", "batch1_forward", "cuda(gfx1100)",
                         "pytorch-rocm", batch=1, warmup=3, iters=ITERS,
                         notes="GPU forward only"), b1)
sumb8 = save_run(RunMeta("labD_batch", "batch8_forward", "cuda(gfx1100)",
                         "pytorch-rocm", batch=8, warmup=3, iters=ITERS,
                         notes="GPU forward only, whole batch timed"), b8)
print(f"batch=1 : P50 {pct(b1,50):.1f} ms · {thr1:.1f} img/s · per-request latency = P50")
print(f"batch=8 : P50 {pct(b8,50):.1f} ms/batch · {thr8:.1f} img/s · "
      f"equivalent {pct(b8,50)/8:.1f} ms/img (NOT request latency)")
print(f"VRAM peak during batches: {vram_batch:.2f} GB")

[saved] results/raw/labD_batch/labD_batch_batch1_forward_20261108_112916.csv  (10 samples)


[saved] results/raw/labD_batch/labD_batch_batch8_forward_20261108_112918.csv  (10 samples)
batch=1 : P50 9.4 ms · 106.1 img/s · per-request latency = P50
batch=8 : P50 55.0 ms/batch · 145.4 img/s · equivalent 6.9 ms/img (NOT request latency)
VRAM peak during batches: 0.98 GB


In [6]:
from course.charts import chart_batch
chart_batch(1, 8, thr1, thr8, f"{pct(b1,50):.0f} ms/req",
            f"{pct(b8,50):.0f} ms/batch", vram640, vram_batch, sumb8)

[chart] /workspace/yolo26-openvino-rocm-day2-labs/assets/slides/05_batch_vs_throughput.png (+svg)


## Common mistakes / 常见错误
- 把"批耗时 ÷ 8"说成用户延迟：用户等到的是**整批完成**，不是 1/8。
  *batch-time/8 is not user latency* — a request waits for the WHOLE batch.
- 只降分辨率不看 mAP：速度收益常常以小目标召回为代价。
  *Resolution gains often cost small-object recall* — always pair latency with mAP.
- 显存只看模型大小：峰值显存主要由激活值（batch × 分辨率²）决定。
  *Peak VRAM is dominated by activations* (batch × resolution²), not just weights.


## Why? / 为什么会这样？
- **分辨率**：640→512 让像素数减少 36%，卷积计算量近似同比例下降；但小目标特征变弱，mAP 通常下降。分辨率是“速度—精度”旋钮。
- **Batch**：GPU 一次并行算 8 张，利用率更高、吞吐上升；但显存近似线性增长，且单个请求必须等待整批（或超时），**延迟口径**变了。

- *Resolution*: 640→512 cuts pixels by 36% and compute similarly; small objects suffer, mAP usually drops. It is the speed-accuracy knob.
- *Batch*: the GPU computes 8 images in parallel → higher utilization & throughput; VRAM grows ~linearly, and a request must wait for its batch — the *latency semantics* changed.

## Try it yourself / 学生练习
回到上面把 `STUDENT_IMGSZ_B` 改成 `416`，重新运行实验 A：延迟、mAP、显存各会怎么变？先预测再验证。
Go back, set `STUDENT_IMGSZ_B = 416`, re-run Experiment A — predict latency / mAP / VRAM first, then verify.

**Takeaway / 一句话总结**
> GPU 优化不是无脑“更大 batch、更小图片”，而是在延迟、吞吐、显存、精度之间找到匹配业务目标的组合。
> GPU tuning is not 'bigger batch, smaller images' — it is finding the > combination that matches your latency, throughput, memory and accuracy > targets.